# Few-shot Discord event announcement (`ai_query`)

**Purpose:** Build a Japanese Discord-style event announcement using the few-shot prompt layout from `docs/few_shot_discord_event_announcement_samples.md`, then call Databricks **`ai_query`** on a Foundation Model endpoint.

**Author:** Cheng Wang  
**Contact:** cheng.wang@myteam.com  
**Date:** 2026-03-23  

## Prerequisites

- Workspace in a region where [AI Functions / `ai_query`](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_query) are available.
- Your user or job identity has **CAN QUERY** on the configured foundation endpoint.
- Cluster / notebook uses a **Databricks Runtime** that supports `ai_query`. **Spark Connect (typical for Jobs):** named arguments `modelParameters` / `failOnError` on `ai_query` are not supported in analyzed SQL (42703) — the last cell uses **2-arg** `ai_query(endpoint, prompt)` only; `TEMPERATURE` / `MAX_TOKENS` in the constants cell are **not applied** to the model call on that path.
- Adjust **`FOUNDATION_ENDPOINT`**, **`TEMPERATURE`**, and **`MAX_TOKENS`** in the constants cell if your workspace uses different defaults.

## Prompt source

Static few-shot core is loaded from **`apps/discord-platform-jedai/server/static_prompt_core.ts`** via `load_static_prompt_core.py` (same order as App `build_prompt.ts`: Hyperparameter → few-shot → context → user request).  
`Source: config/prompt/few_shot_discord_event_announcement_samples.md` (synced into the App TS bundle).


## Web app contract (later)

| Notebook widget   | Client (form / JSON)     |
|-------------------|--------------------------|
| `tone`            | Same six hyperparameters |
| `length`          | as dropdown fields       |
| `formality`       |                          |
| `emoji_density`   |                          |
| `structure`       |                          |
| `cta_strength`    |                          |
| `user_request`    | Natural-language field   |

**Server-side only (not from client):** context facts (`CONTEXT_FOR_REQUEST`), model endpoint name, `TEMPERATURE`, `MAX_TOKENS` — same as the constants in this notebook. Build the full prompt server-side, then run one `ai_query` row (or equivalent SQL / Jobs API).


In [ ]:
# --- Fixed in code (edit per environment / event) ---
# Source: docs/few_shot_discord_event_announcement_samples.md

CONTEXT_FOR_REQUEST = """- 日時: 5月24日（土）21:00
- ゲーム: Geoguessr
- 場所: TitanZz Discord
- 参加方法: この投稿にリアクション
"""

FOUNDATION_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
TEMPERATURE = 0.3
MAX_TOKENS = 2048


In [ ]:
# Widgets: six behavior hyperparameters + natural language request
dbutils.widgets.dropdown("tone", "カジュアル", ["真面目", "おふざけ", "カジュアル"], "Tone")
dbutils.widgets.dropdown("length", "medium", ["short", "medium", "long"], "Length")
dbutils.widgets.dropdown("formality", "ですます", ["ですます", "タメ口", "敬語"], "Formality")
dbutils.widgets.dropdown("emoji_density", "普通", ["なし", "少なめ", "普通", "多め"], "Emoji density")
dbutils.widgets.dropdown("structure", "箇条書き中心", ["箇条書き中心", "段落中心", "見出し＋本文"], "Structure")
dbutils.widgets.dropdown("cta_strength", "普通", ["控えめ", "普通", "強め"], "CTA strength")
dbutils.widgets.text(
    "user_request",
    "来週土曜21時の練習会告知を、カジュアルで中くらいの長さ、箇条書き中心で作って",
    "User request",
)

tone = dbutils.widgets.get("tone")
length = dbutils.widgets.get("length")
formality = dbutils.widgets.get("formality")
emoji_density = dbutils.widgets.get("emoji_density")
structure = dbutils.widgets.get("structure")
cta_strength = dbutils.widgets.get("cta_strength")
user_request = dbutils.widgets.get("user_request")


In [ ]:
# App-aligned static prompt (apps/discord-platform-jedai/server/static_prompt_core.ts)
import sys
from pathlib import Path

_repo = Path.cwd()
while _repo != _repo.parent and not (_repo / "apps" / "discord-platform-jedai").exists():
    _repo = _repo.parent

_models_dir = _repo / "scripts" / "05_models"
if str(_models_dir) not in sys.path:
    sys.path.insert(0, str(_models_dir))

from load_static_prompt_core import load_static_prompt_core

STATIC_PROMPT_CORE = load_static_prompt_core()

assert "[System / Role]" in STATIC_PROMPT_CORE
assert "[Single-parameter Examples" in STATIC_PROMPT_CORE
assert "[Output instruction]" in STATIC_PROMPT_CORE
assert "Few-shot examples below are for tone" in STATIC_PROMPT_CORE
assert len(STATIC_PROMPT_CORE) > 10_000, f"prompt too short: {len(STATIC_PROMPT_CORE)}"

print(f"STATIC_PROMPT_CORE loaded ({len(STATIC_PROMPT_CORE)} chars) from App static_prompt_core.ts")


In [ ]:
from load_static_prompt_core import build_full_prompt

full_prompt = build_full_prompt(
    tone=tone,
    length=length,
    formality=formality,
    emoji_density=emoji_density,
    structure=structure,
    cta_strength=cta_strength,
    user_request=user_request,
    context_facts=CONTEXT_FOR_REQUEST,
    static_core=STATIC_PROMPT_CORE,
)

_hyper_idx = full_prompt.index(f"Emoji density: {emoji_density}")
_few_shot_idx = full_prompt.index("[Single-parameter Examples")
assert _hyper_idx < _few_shot_idx, "Hyperparameter block must precede few-shot examples"
assert "Emoji density: なし → use zero emoji" in full_prompt or emoji_density != "なし"
print(f"full_prompt built ({len(full_prompt)} chars); hyper before few-shot: OK")


In [ ]:
df = spark.createDataFrame([(full_prompt,)], ["full_prompt"])

# Spark Connect (incl. many Job runs): `modelParameters` / `failOnError` are not resolved inside
# ai_query(...) — same failure for selectExpr and spark.sql (SQLSTATE 42703). Use 2-arg form;
# decoding follows the foundation endpoint defaults (constants TEMPERATURE / MAX_TOKENS above are intent only).
result = df.selectExpr(f"ai_query('{FOUNDATION_ENDPOINT}', full_prompt) AS announcement")

display(result)
